# ECE-GY 9143 Lab 4 Part A: Distributed Deep Learning (Cloud Pod)

This notebook is adapted for cloud multi-GPU pods (RunPod/Lambda style environments).
It executes experiments by calling `lab4_ddp.py` through `%%bash` cells.

## Before running
- Confirm this pod has 4 GPUs available.
- Open JupyterLab terminal in the project folder (where `lab4_ddp.py` is located).
- Keep default Lab 2 hyper-parameters and `num_workers=2`.

## Config defaults used in code cells
- `PROJECT_DIR=$PWD`
- `DATA_PATH=/workspace/data/cifar10`
- `LOG_DIR=$PROJECT_DIR/logs`

You can override these with environment variables inside each cell if needed.


In [1]:
%%bash
set -euo pipefail
python3 -c "import torch; print('torch:', torch.__version__); print('cuda_available:', torch.cuda.is_available()); print('gpu_count:', torch.cuda.device_count())"
nvidia-smi


torch: 2.4.1+cu124
cuda_available: True
gpu_count: 4
Wed Apr 15 22:05:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   33C    P0             71W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabl

## Q1: Computational Efficiency vs Batch Size (Single GPU)

Run 2 epochs per batch size and report the **2nd epoch `train_time`** from `Q1 RESULT`.
Start from 32 and increase by 4x until OOM.


In [2]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
mkdir -p "$DATA_PATH" "$LOG_DIR"
cd "$PROJECT_DIR"

# Increase by 4x until OOM; the loop stops at first failure.
for bs in 32 128 512 2048 8192 32768 131072; do
  echo "=== Q1 | batch_size=${bs} ==="
  if ! python3 lab4_ddp.py q1 --batch_size "$bs" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q1_bs${bs}.log"; then
    echo "Q1 stopped at batch_size=${bs} (likely OOM). Use the previous successful batch size as MAX_BS."
    break
  fi
done


=== Q1 | batch_size=32 ===
Q1: Computational Efficiency w.r.t Batch Size (Single GPU)
GPU: NVIDIA A100-SXM4-80GB
Batch size: 32
Files already downloaded and verified
Model parameters: 11,173,962
Number of batches per epoch: 1563

--- Epoch 1 (warmup) ---
  batch  200/1563 | loss 2.1953 | top1 12.50%
  batch  400/1563 | loss 2.0931 | top1 25.00%
  batch  600/1563 | loss 1.9593 | top1 28.12%
  batch  800/1563 | loss 1.7833 | top1 34.38%
  batch 1000/1563 | loss 1.7915 | top1 40.62%
  batch 1200/1563 | loss 1.7442 | top1 40.62%
  batch 1400/1563 | loss 1.6098 | top1 37.50%
  batch 1563/1563 | loss 1.4726 | top1 43.75%
  Data loading time : 0.238s
  Training time     : 12.935s  <-- Q1 metric (excl data I/O)
  Total epoch time  : 13.757s
  Avg loss          : 1.9586
  Avg top-1 acc     : 28.69%

--- Epoch 2 (measured) ---
  batch  200/1563 | loss 1.2914 | top1 56.25%
  batch  400/1563 | loss 1.6584 | top1 40.62%
  batch  600/1563 | loss 1.7595 | top1 25.00%
  batch  800/1563 | loss 1.9647 |

## Q2: Speedup Measurement (2 GPU / 4 GPU)

For each batch size used in Q1 (e.g., 32/128/512), run 2 epochs and report the **2nd epoch `total_time`**.
Then compute speedup using `Speedup = T(1-GPU total_time) / T(N-GPU total_time)` for each setup.

> This is weak scaling because batch size per GPU is fixed while effective batch size increases with GPU count.


In [3]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
BATCHES_STR=${BATCHES_STR:-"32 128 512"}
mkdir -p "$DATA_PATH" "$LOG_DIR"
cd "$PROJECT_DIR"

for bs in $BATCHES_STR; do
  echo "=== Q2 | 2 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=2 lab4_ddp.py q2 --batch_size "$bs" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g2_bs${bs}.log"

  echo "=== Q2 | 4 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=4 lab4_ddp.py q2 --batch_size "$bs" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g4_bs${bs}.log"
done


=== Q2 | 2 GPUs | batch_size_per_gpu=32 ===
W0415 22:07:20.346000 127192902012928 torch/distributed/run.py:779] 
W0415 22:07:20.346000 127192902012928 torch/distributed/run.py:779] *****************************************
W0415 22:07:20.346000 127192902012928 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 22:07:20.346000 127192902012928 torch/distributed/run.py:779] *****************************************
Q2: Speedup Measurement (DDP)
GPU: NVIDIA A100-SXM4-80GB
World size (num GPUs): 2
Batch size per GPU: 32
Effective batch size: 64
Files already downloaded and verified
Files already downloaded and verified
Batches per GPU per epoch: 782

--- Epoch 1 (warmup) ---
  batch  200/782 | loss 1.9453 | top1 34.38%
  batch  400/782 | loss 1.8463 | top1 25.00%
  batch  600/782 | loss 1.3358 | top

In [4]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
MAX_BS=${MAX_BS:-8192}
mkdir -p "$LOG_DIR"
cd "$PROJECT_DIR"

torchrun --nproc_per_node=2 lab4_ddp.py q2 --batch_size "$MAX_BS" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g2_bs${MAX_BS}.log"
torchrun --nproc_per_node=4 lab4_ddp.py q2 --batch_size "$MAX_BS" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g4_bs${MAX_BS}.log"


W0415 22:09:11.566000 136331194175488 torch/distributed/run.py:779] 
W0415 22:09:11.566000 136331194175488 torch/distributed/run.py:779] *****************************************
W0415 22:09:11.566000 136331194175488 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 22:09:11.566000 136331194175488 torch/distributed/run.py:779] *****************************************
Q2: Speedup Measurement (DDP)
GPU: NVIDIA A100-SXM4-80GB
World size (num GPUs): 2
Batch size per GPU: 8192
Effective batch size: 16384
Files already downloaded and verified
Files already downloaded and verified
Batches per GPU per epoch: 4

--- Epoch 1 (warmup) ---
  batch    4/4 | loss 3.4595 | top1 15.09%
  Data loading time : 2.088s
  Training time     : 2.513s
  Total epoch time  : 4.703s  <-- Q2 metric (incl all)
  Avg loss 

## Q3: Computation vs Communication + Bandwidth Utilization

Run 2-GPU and 4-GPU for each batch size. Report the 2nd epoch `train_time` from Q3.

For each `(num_gpu, batch_size_per_gpu)`:
- `compute_time = Q1 train_time`
- `comm_time = Q3 DDP train_time - Q1 train_time`

For Q3.2 bandwidth utilization, use:
- `model_size_bytes = num_params * 4`
- `allreduce_volume = 2 * (N-1)/N * model_size_bytes`
- `effective_bandwidth = allreduce_volume / comm_time_per_iter`


In [5]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
BATCHES_STR=${BATCHES_STR:-"32 128 512"}
mkdir -p "$DATA_PATH" "$LOG_DIR"
cd "$PROJECT_DIR"

for bs in $BATCHES_STR; do
  echo "=== Q3 | 2 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=2 lab4_ddp.py q3 --batch_size "$bs" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q3_g2_bs${bs}.log"

  echo "=== Q3 | 4 GPUs | batch_size_per_gpu=${bs} ==="
  torchrun --nproc_per_node=4 lab4_ddp.py q3 --batch_size "$bs" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q3_g4_bs${bs}.log"
done


=== Q3 | 2 GPUs | batch_size_per_gpu=32 ===
W0415 22:09:47.221000 127479537557504 torch/distributed/run.py:779] 
W0415 22:09:47.221000 127479537557504 torch/distributed/run.py:779] *****************************************
W0415 22:09:47.221000 127479537557504 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 22:09:47.221000 127479537557504 torch/distributed/run.py:779] *****************************************
Q3: Computation vs Communication (DDP)
GPU: NVIDIA A100-SXM4-80GB
World size: 2
Batch size per GPU: 32
Files already downloaded and verified
Files already downloaded and verified

--- Epoch 1 (warmup) ---
  batch  200/782 | loss 1.9956 | top1 15.62%
  batch  400/782 | loss 1.8521 | top1 37.50%
  batch  600/782 | loss 1.3624 | top1 50.00%
  batch  782/782 | loss 1.8241 | top1 37.50%
  Tr

In [6]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
MAX_BS=${MAX_BS:-8192}
mkdir -p "$LOG_DIR"
cd "$PROJECT_DIR"

torchrun --nproc_per_node=2 lab4_ddp.py q3 --batch_size "$MAX_BS" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g2_bs${MAX_BS}.log"
torchrun --nproc_per_node=4 lab4_ddp.py q3 --batch_size "$MAX_BS" --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q2_g4_bs${MAX_BS}.log"


W0415 22:11:33.251000 129739423367168 torch/distributed/run.py:779] 
W0415 22:11:33.251000 129739423367168 torch/distributed/run.py:779] *****************************************
W0415 22:11:33.251000 129739423367168 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 22:11:33.251000 129739423367168 torch/distributed/run.py:779] *****************************************
Q3: Computation vs Communication (DDP)
GPU: NVIDIA A100-SXM4-80GB
World size: 2
Batch size per GPU: 8192
Files already downloaded and verified
Files already downloaded and verified

--- Epoch 1 (warmup) ---
  batch    4/4 | loss 2.8789 | top1 20.75%
  Train time (compute+comm): 2.373s
  Total epoch time          : 4.763s

--- Epoch 2 (measured) ---
  batch    4/4 | loss 3.4748 | top1 11.08%
  Train time (compute+comm): 2.231s
  T

## Q4.1: Large Batch Training (4 GPUs, 5 Epochs)

Use the largest successful `batch_size_per_gpu` from Q1 (`MAX_BS`).
Report 5th epoch average loss and accuracy, then compare with Lab 2 (`batch_size=128`, single GPU).


In [7]:
%%bash
set -euo pipefail
PROJECT_DIR=${PROJECT_DIR:-$PWD}
DATA_PATH=${DATA_PATH:-/workspace/data/cifar10}
LOG_DIR=${LOG_DIR:-$PROJECT_DIR/logs}
MAX_BS=${MAX_BS:-8192}   
mkdir -p "$DATA_PATH" "$LOG_DIR"
cd "$PROJECT_DIR"

echo "=== Q4 | 4 GPUs | batch_size_per_gpu=${MAX_BS} | epochs=5 ==="
torchrun --nproc_per_node=4 lab4_ddp.py q4 --batch_size "$MAX_BS" --epochs 5 --data_path "$DATA_PATH" 2>&1 | tee "$LOG_DIR/q4_bs${MAX_BS}.log"


=== Q4 | 4 GPUs | batch_size_per_gpu=8192 | epochs=5 ===
W0415 22:12:09.149000 130539119640576 torch/distributed/run.py:779] 
W0415 22:12:09.149000 130539119640576 torch/distributed/run.py:779] *****************************************
W0415 22:12:09.149000 130539119640576 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 22:12:09.149000 130539119640576 torch/distributed/run.py:779] *****************************************
Q4: Large Batch Training (DDP)
GPU: NVIDIA A100-SXM4-80GB
World size: 4
Batch size per GPU: 8192
Effective batch size: 32768
Epochs: 5
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified

--- Epoch 1/5 ---
  batch    2/2 | loss 2.5359 | top1 18.06%
  Avg loss  : 2.4336
  Avg 

## Q4.2: How to Improve Training Accuracy for Large Batch (Report Section)

Based on reference [4], two effective remedies are:

1. **Linear learning-rate scaling with warmup**  
   When the global batch size is increased by a factor of $k$, scale the learning rate by $k$:

   $$
   \eta' = k\eta
   $$

   Then use **gradual warmup** in early epochs to avoid instability at the start of training.

2. **Careful BatchNorm handling in distributed training**  
   Large-batch training is sensitive to BatchNorm statistics. Following [4], maintain consistent BN behavior across workers (instead of naively changing BN statistics aggregation). This improves optimization stability and final accuracy.

---


## Q5: What Is Passed on Network? (Report Section)

**No, gradients are not the only communicated data.**

In PyTorch DDP, gradient all-reduce is the main per-iteration communication, but additional synchronization also occurs for model consistency, including initial parameter synchronization and buffer synchronization (e.g., BatchNorm running statistics, depending on configuration). Therefore, gradients are not the only messages exchanged.

---


## Q6: What If We Only Communicate Gradients? (Report Section)

**Communicating only gradients is not sufficient to guarantee good accuracy.**

Gradient synchronization is necessary, but large-batch accuracy also depends on training strategy from [4], especially learning-rate scaling, warmup, and proper BatchNorm treatment. Without these, training may remain synchronized yet still converge to worse accuracy.